[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SivanSchwartz/representation-learning-course/blob/master/lec5/notebooks/05_moco_experiments.ipynb)

# MoCo Experiments: Exercises
### Representation Learning, Lecture 5: Contrastive Learning II, Momentum Contrast

This notebook runs experiments with the MoCo code from `model.py`. Most exercises follow the same order: **predict**, run a short experiment, then explain what you saw.

**Recap.** A query $q$ and a key $k_+$ come from two augmented views of the same image; $q$ is computed by the query network $\theta_q$ (encoder + projector) and $k_+$ by the key network $\theta_k$. Both are $\ell_2$-normalized. The queue holds $K$ keys $\{k_i\}_{i=1}^{K}$ from earlier batches. The loss is a $(K{+}1)$-way cross-entropy with the positive in position 0:

$$
\mathcal{L}_q=-\log\frac{\exp(q\cdot k_+/\tau)}{\exp(q\cdot k_+/\tau)+\sum_{i=1}^{K}\exp(q\cdot k_i/\tau)}
$$

Only $\theta_q$ receives gradients. The key network follows it as an exponential moving average (EMA):

$$
\theta_k\leftarrow m\,\theta_k+(1-m)\,\theta_q
$$

| symbol | code | value |
|---|---|---|
| $K$ | `config.MOCO_K` | 4096 |
| $m$ | `config.MOCO_M` | 0.999 |
| $\tau$ | `config.TEMPERATURE` (`T` in `MoCo`) | 0.07 |
| $B$ | `config.BATCH_SIZE` | 64 |
| $d$ | `config.PROJECTOR_DIM` | 128 |

**How to work.** Cells marked `# TODO` are yours to complete. Questions are marked **R** (reading the code) and **Q** (experiments). Write your prediction *before* you run the cell below a question.

**Runtime.** Every training run is short (one or two epochs of a small encoder). The whole notebook ran in about 40 minutes on a 2-core CPU without a GPU; a Colab GPU should be faster. With `EPOCHS = 1` it takes about half as long.

## 0. Setup
The notebook imports the lecture code (`model.py`, `augmentations.py`, `config.py`). On Colab, the next cell downloads the `lec5` folder from the course repository; locally, it uses the parent folder of `notebooks/`. In Colab, choose *Runtime → Change runtime type → T4 GPU* first.

In [ ]:
import os
import sys

LECTURE = "lec5"
REPO = "/content/representation-learning-course"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Download only this lecture's folder from the course repository.
    if not os.path.exists(REPO):
        !git clone --filter=blob:none --sparse https://github.com/SivanSchwartz/representation-learning-course.git {REPO}
    os.chdir(REPO)
    !git sparse-checkout set {LECTURE}
    os.chdir(f"{REPO}/{LECTURE}")
    !pip install -q pytorch-lightning torchmetrics
    CODE_DIR = "."   # Colab: we are inside lec5
else:
    CODE_DIR = ".."  # locally: this notebook sits in lec5/notebooks

sys.path.insert(0, os.path.abspath(CODE_DIR))
print("code folder:", os.path.abspath(CODE_DIR))

In [ ]:
import copy
import logging
import math
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pytorch_lightning as pl
import torch
import torch.nn.functional as F
import torchvision
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets
from torchvision.transforms import v2

import config
from augmentations import ContrastiveTransformations, build_simclr_transform
from model import MoCo, SimCLR, AutoEncoder, SupervisedEncoder

# Short output: no Lightning banners and no known harmless warnings.
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", ".*val_dataloader.*")
warnings.filterwarnings("ignore", ".*does not have many workers.*")
warnings.filterwarnings("ignore", ".*LeafSpec.*")
torch.set_float32_matmul_precision("medium")

In [ ]:
DATASET = "FashionMNIST"  # "MNIST" or "FashionMNIST". Exercise 6 always uses MNIST.
DATA_DIR = os.path.join(CODE_DIR, config.DATA_DIR)
B = config.BATCH_SIZE      # 64
D = config.PROJECTOR_DIM   # 128, the length of q and k
EPOCHS = 2                 # per pretraining run; 1 is enough to see every effect
NUM_WORKERS = min(4, os.cpu_count() or 0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PRECISION = "16-mixed" if DEVICE == "cuda" else "32-true"

results = {}  # name -> {"linear_probe": accuracy, "knn": accuracy}, filled as we go
print(f"{DATASET}: batch {B}, queue {config.MOCO_K}, m {config.MOCO_M}, T {config.TEMPERATURE}, device {DEVICE}")

### Helpers
* `contrastive_loader` is `MnistDataModule(two_views=True)` written out, so that we can swap the augmentation (Exercise 6).
* `fit` is the `Trainer` from `train.py`, without validation. Every run starts from the same seed. The runs are logged to `tb_logs/lec5_experiments` (optional: `%load_ext tensorboard` and `%tensorboard --logdir tb_logs`).
* `evaluate` freezes an encoder and reports two test accuracies, using only `N_LABELS = 1000` labeled training images (the same 1000 every time). Few labels and many unlabeled images is the situation self-supervised learning is for; set `N_LABELS = None` to use all 60,000 labels.
  * **linear probe**: logistic regression on the frozen features. This is `--mode linear_probe`, but the features are computed once, so it takes seconds.
  * **kNN** ($k=10$, cosine similarity): the majority label of the 10 most similar labeled images. No training at all.

In [ ]:
EVAL_TRANSFORM = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize((0.1307,), (0.3081,)),
])


def contrastive_loader(dataset=DATASET, batch_size=B, view_transform=None):
    """Batches of ((x_q, x_k), y): two augmented views of each training image."""
    if view_transform is None:
        view_transform = build_simclr_transform()
    ds = getattr(datasets, dataset)(DATA_DIR, train=True, download=True,
                                    transform=ContrastiveTransformations(view_transform))
    # drop_last: the queue pointer must move by exactly batch_size every step
    return DataLoader(ds, batch_size=batch_size, shuffle=True, drop_last=True, num_workers=NUM_WORKERS)


def plain_loader(dataset=DATASET, train=True, shuffle=False, batch_size=512, indices=None):
    """Batches of (x, y): one image, no augmentation. indices: use only these images."""
    ds = getattr(datasets, dataset)(DATA_DIR, train=train, download=True, transform=EVAL_TRANSFORM)
    if indices is not None:
        ds = Subset(ds, indices)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=NUM_WORKERS)


class EpochPrinter(pl.Callback):
    """One line per epoch, instead of a progress bar."""

    def on_train_epoch_start(self, trainer, pl_module):
        self.start = time.time()

    def on_train_epoch_end(self, trainer, pl_module):
        metrics = trainer.callback_metrics
        loss = metrics.get("train_loss_epoch", metrics.get("train_loss"))
        print(f"  epoch {trainer.current_epoch}: train_loss {loss:.3f} ({time.time() - self.start:.0f}s)")


def fit(model, loader, name, epochs=EPOCHS, max_steps=-1, callbacks=()):
    """Train with Lightning and return the model (on the CPU)."""
    pl.seed_everything(42, verbose=False)  # same data order in every run
    print(f"{name}:")
    trainer = pl.Trainer(
        max_epochs=epochs,
        max_steps=max_steps,
        accelerator="auto",
        devices=1,
        precision=PRECISION,
        logger=pl.loggers.TensorBoardLogger("tb_logs", name="lec5_experiments", version=name),
        callbacks=[EpochPrinter(), *callbacks],
        log_every_n_steps=10,
        enable_progress_bar=False,
        enable_model_summary=False,
        enable_checkpointing=False,
    )
    trainer.fit(model, train_dataloaders=loader)
    return model.cpu()

In [ ]:
@torch.no_grad()
def extract_features(encoder, loader):
    encoder = encoder.to(DEVICE).eval()
    feats, labels = [], []
    for x, y in loader:
        feats.append(encoder(x.to(DEVICE)).float().cpu())
        labels.append(y)
    return torch.cat(feats), torch.cat(labels)


def linear_probe(train_x, train_y, test_x, test_y, steps=300):
    """Logistic regression on frozen features (full batch, standardized features)."""
    mean, std = train_x.mean(0), train_x.std(0) + 1e-6
    train_x, test_x = (train_x - mean) / std, (test_x - mean) / std
    torch.manual_seed(0)  # same initial head in every evaluation
    head = nn.Linear(train_x.shape[1], int(train_y.max()) + 1)
    optimizer = torch.optim.Adam(head.parameters(), lr=0.01)
    for _ in range(steps):
        optimizer.zero_grad()
        F.cross_entropy(head(train_x), train_y).backward()
        optimizer.step()
    return (head(test_x).argmax(1) == test_y).float().mean().item()


def knn_accuracy(train_x, train_y, test_x, test_y, k=10):
    """Majority label of the k most similar training images (cosine similarity)."""
    train_x, test_x = F.normalize(train_x, dim=1), F.normalize(test_x, dim=1)
    correct = 0
    for i in range(0, len(test_x), 500):
        nearest = (test_x[i:i + 500] @ train_x.T).topk(k, dim=1).indices  # [500, k]
        pred = train_y[nearest].mode(dim=1).values
        correct += (pred == test_y[i:i + 500]).sum().item()
    return correct / len(test_x)


N_LABELS = 1000
LABELED = None if N_LABELS is None else torch.randperm(60_000, generator=torch.Generator().manual_seed(0))[:N_LABELS]


def evaluate(encoder, name, dataset=DATASET):
    """Linear-probe and kNN test accuracy of a frozen encoder, with N_LABELS labels. Stored in results[name]."""
    train_x, train_y = extract_features(encoder, plain_loader(dataset, train=True, indices=LABELED))
    test_x, test_y = extract_features(encoder, plain_loader(dataset, train=False))
    results[name] = {
        "linear_probe": linear_probe(train_x, train_y, test_x, test_y),
        "knn": knn_accuracy(train_x, train_y, test_x, test_y),
    }
    print(f"{name:<32} linear probe {results[name]['linear_probe']:.3f}   kNN {results[name]['knn']:.3f}")


def show_grid(images, title, nrow=8):
    grid = torchvision.utils.make_grid(images, nrow=nrow, padding=2, pad_value=1.0, normalize=True)
    plt.figure(figsize=(nrow, nrow * grid.shape[1] / grid.shape[2]))
    plt.imshow(grid.permute(1, 2, 0))  # CHW -> HWC for matplotlib
    plt.axis("off")
    plt.title(title)
    plt.show()


def smooth(values, n=25):
    """Moving average, for noisy per-step curves."""
    return np.convolve(values, np.ones(n) / n, mode="valid")

## 1. Reading the code
Open `model.py` at `class MoCo`. Answer these questions before you run anything.

**R1.** `__init__` builds the key network with `copy.deepcopy(self.encoder)` and `copy.deepcopy(self.projector)`. Why must it start as an exact copy of the query network? What would go wrong with an independent random initialization?

*Your answer:*

**R2.** The queue is stored with `register_buffer`. What would change if it were **(a)** a plain attribute, `self.queue = torch.randn(...)`, or **(b)** an `nn.Parameter`?

*Your answer:*

**R3.** The keys are computed inside `with torch.no_grad()`. Give two reasons.

*Your answer:*

**R4.** `_dequeue_and_enqueue(k)` runs *after* the logits are computed. Suppose we enqueue first. Which key now appears twice in the logits, and what is the smallest loss the model can reach?

*Your answer:*

**R5.** Why does `_dequeue_and_enqueue` check `assert self.K % batch_size == 0`, and why does `MnistDataModule` use `drop_last=True` for two views?

*Your answer:*

**R6.** `validation_step` also calls `_common_step`. Which two operations run only when `self.training` is `True`, and why must validation skip them?

*Your answer:*

**R7.** `configure_optimizers` gives Adam only the query encoder and projector. What would happen with `optim.Adam(self.parameters())`?

*Your answer:*

**R8.** `train.py` saves `model.encoder` for the linear probe. Why not the projector? Could we save `model.key_encoder` instead?

*Your answer:*

**R9.** MoCo v1 needed *Shuffling BN*. Why? Why does our model not need it?

*Your answer:*

**R10.** The code has a note for you: *why do we normalize the output of the projector?*

*Your answer:*

**R11.** Cost. Count the multiply-adds of `q @ self.queue` for $B=64$, $K=4096$, $d=128$, and compare with one forward pass of the encoder and projector on a batch. How large would a SimCLR batch need to be for 4096 negatives per anchor?

*Your answer:*

### Two quick checks
**Check A.** The code writes the loss as `-(logits[:, 0] - logsumexp(logits)).mean()`. This is cross-entropy with the label 0 for every query, $-\log\operatorname{softmax}(\text{logits})_0$. Confirm it with `F.cross_entropy`.

In [ ]:
torch.manual_seed(0)
logits = torch.randn(B, 1 + config.MOCO_K) / config.TEMPERATURE  # stand-in logits, [B, 1+K]
moco_loss = -(logits[:, 0] - torch.logsumexp(logits, dim=1)).mean()
ce_loss = ...  # TODO: F.cross_entropy, with the positive as the target class
print(f"MoCo loss {moco_loss:.4f}   cross-entropy {ce_loss:.4f}")

**Check B.** At step 0 the queue is `F.normalize(torch.randn(d, K), dim=0)`: $K$ random unit vectors. For a fixed unit vector $q$ and a random unit vector $u\in\mathbb{R}^d$,
$$\mathbb{E}[q\cdot u]=0,\qquad \operatorname{Var}(q\cdot u)=\frac1d .$$
Check this numerically for $d=128$.

In [ ]:
q = F.normalize(torch.randn(D), dim=0)
u = F.normalize(torch.randn(D, 100_000), dim=0)  # 100k random unit vectors, like the initial queue
dots = ...  # TODO: the dot product of q with every column of u
print(f"mean {dots.mean():.4f}   std {dots.std():.4f}   1/sqrt(d) = {1 / math.sqrt(D):.4f}")

**R12.** At step 0, what is the typical size of a negative logit $q\cdot u/\tau$ with $\tau=0.07$? Is the initial random queue a hard or an easy set of negatives?

*Your answer:*

## Exercise 1: What does the loss do in the first steps?
At step 0 the queue holds random vectors. After step $t$ it holds $Bt$ real keys (computed by the key network) and $K-Bt$ random vectors, until it is full at step $K/B$.

First, measure similarities at initialization, before any training.

In [ ]:
pl.seed_everything(42, verbose=False)
moco0 = MoCo(learning_rate=config.LEARNING_RATE, latent_dim=config.LATENT_DIM, projector_dim=D,
             K=config.MOCO_K, m=config.MOCO_M, T=config.TEMPERATURE)
(x_q, x_k), y = next(iter(contrastive_loader()))

with torch.no_grad():
    q = F.normalize(moco0.projector(moco0.encoder(x_q)), dim=1)          # [B, d]
    k = F.normalize(moco0.key_projector(moco0.key_encoder(x_k)), dim=1)  # [B, d]
    pos = (q * k).sum(dim=1)                       # q . k+, same image
    other = (q @ k.T)[~torch.eye(B, dtype=torch.bool)]  # q . k of the OTHER images in the batch
    rand = q @ moco0.queue                         # q . u, the random initial queue

print(f"q . k+  (same image, two views): {pos.mean():.3f}")
print(f"q . k   (other images)         : {other.mean():.3f} +- {other.std():.3f}")
print(f"q . u   (random queue)         : {rand.mean():.3f} +- {rand.std():.3f}")

**Q1.1.** At initialization $q\cdot k_+$ is almost the same as $q\cdot k$ for *other* images. What does that tell you about a randomly initialized network?

*Your answer:*

**Q1.2. Prediction.** Predict the loss **(a)** at step 0; **(b)** after $t$ steps, while the queue fills. Assume every real key in the queue is about as similar to $q$ as $k_+$ is, and ignore the random vectors. **(c)** What is the largest value the loss could reach?

*Hint:* if $n$ negatives have the same logit $s$ as the positive and all other negatives are negligible, $\mathcal{L}=-\log\frac{e^{s}}{e^{s}+n\,e^{s}}$.

*Your answer:*

Now train for 150 steps with two queue sizes and compare with your prediction. `LossRecorder` stores the loss after every step.

In [ ]:
class LossRecorder(pl.Callback):
    """Stores the training loss after every step."""

    def __init__(self):
        self.losses = []

    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
        self.losses.append(outputs["loss"].item())


curves = {}
for K in [1024, 4096]:
    pl.seed_everything(42, verbose=False)
    model = MoCo(learning_rate=config.LEARNING_RATE, latent_dim=config.LATENT_DIM, projector_dim=D,
                 K=K, m=config.MOCO_M, T=config.TEMPERATURE)
    recorder = LossRecorder()
    fit(model, contrastive_loader(), name=f"ex1_K{K}", max_steps=150, callbacks=[recorder])
    curves[K] = recorder.losses

In [ ]:
steps = np.arange(150)
plt.figure(figsize=(9, 4))
for K, losses in curves.items():
    line, = plt.plot(losses, label=f"measured, K={K}")
    prediction = ...  # TODO: your formula from Q1.2(b); the queue cannot hold more than K keys
    plt.plot(steps, prediction, "--", color=line.get_color(), label=f"prediction, K={K}")
    plt.axvline(K // B, ls=":", color=line.get_color())
plt.xlabel("training step t")
plt.ylabel("loss")
plt.ylim(0, 9)
plt.legend()
plt.title("MoCo loss in the first 150 steps (dotted: the queue is full)")
plt.show()

**Q1.3.** Where does the measured curve follow your prediction, and where does it leave it? Why does it leave it?

*Your answer:*

**Q1.4.** The two curves almost coincide during the first steps. Why? What changes for $K=1024$ at step 16?

*Your answer:*

**Q1.5.** Can you compare the loss *values* of runs with different $K$, or of MoCo with SimCLR? What would you compare instead?

*Your answer:*

## Exercise 2: Measuring what MoCo learns
`SimCLR` in `model.py` logs `pos_sim`, `neg_sim` and `acc_top1`; `MoCo` logs only the loss. `MoCoLab` below is `MoCo` with one method rewritten: `_common_step` is copied from `model.py` and gets
* four measurements, logged to TensorBoard and saved in `self.history` after each training step;
* a switch `use_negatives` for Exercise 7.

The measurements:
* `pos_sim` $=\overline{q\cdot k_+}$ and `neg_sim` $=\overline{q\cdot k_i}$ over the queue;
* `acc_top1`: how often the positive has the largest of the $K+1$ logits (chance: $1/(K+1)$);
* `z_std`: the standard deviation of each coordinate of $q$ over the batch, averaged over the $d$ coordinates. For unit vectors spread evenly over the sphere it is $\approx1/\sqrt d$; if all queries are the same vector it is 0.

In [ ]:
class MoCoLab(MoCo):
    """model.MoCo with measurements. Only _common_step changes."""

    def __init__(self, *args, use_negatives=True, **kwargs):
        super().__init__(*args, **kwargs)
        self.use_negatives = use_negatives
        self.history = []  # one dict per training step

    def _common_step(self, batch, mode):
        (im_q, im_k), _ = batch
        q = F.normalize(self.projector(self.encoder(im_q)), dim=1)  # [B, d]
        with torch.no_grad():
            if self.training:
                self._momentum_update_key_encoder()
            k = F.normalize(self.key_projector(self.key_encoder(im_k)), dim=1)  # [B, d]

        l_pos = (q * k).sum(dim=1, keepdim=True)  # [B, 1]
        l_neg = q @ self.queue.clone().detach()  # [B, K]
        logits = torch.cat([l_pos, l_neg], dim=1) / self.T  # [B, 1+K]
        if self.training:
            self._dequeue_and_enqueue(k)

        if self.use_negatives:
            loss = -(logits[:, 0] - torch.logsumexp(logits, dim=1)).mean()
        else:
            loss = ...  # TODO (Exercise 7): the loss with the positive term only

        stats = {
            "pos_sim": ...,   # TODO: mean of q . k+
            "neg_sim": ...,   # TODO: mean of q . k_i over the queue
            "acc_top1": ...,  # TODO: fraction of rows where logit 0 is the largest
            "z_std": ...,     # TODO: std of q over the batch (dim=0), averaged over the d coordinates
        }
        self.log(f"{mode}_loss", loss, on_step=(mode == "train"), on_epoch=True, prog_bar=True)
        self.log_dict({f"{mode}_{name}": value for name, value in stats.items()},
                      on_step=(mode == "train"), on_epoch=True)
        if self.training:
            self.history.append({"loss": loss.item(), **{name: value.item() for name, value in stats.items()}})
        return loss


def make_moco(K=config.MOCO_K, m=config.MOCO_M, use_negatives=True):
    """A new MoCoLab. The seed gives every run the same initial weights."""
    pl.seed_everything(42, verbose=False)
    return MoCoLab(learning_rate=config.LEARNING_RATE, latent_dim=config.LATENT_DIM, projector_dim=D,
                   K=K, m=m, T=config.TEMPERATURE, use_negatives=use_negatives)

Train the baseline with the lecture settings ($K=4096$, $m=0.999$, $\tau=0.07$). We reuse `moco_base` in Exercises 4, 5, 7 and 8. Then compare it with the same encoder before training.

In [ ]:
moco_base = fit(make_moco(), contrastive_loader(), name="moco_base")
evaluate(make_moco().encoder, "random encoder")
evaluate(moco_base.encoder, "MoCo K=4096")

In [ ]:
def plot_history(histories, keys=("loss", "pos_sim", "neg_sim", "acc_top1", "z_std")):
    """histories: {label: model.history}. One panel per measurement."""
    fig, axes = plt.subplots(1, len(keys), figsize=(3.6 * len(keys), 3))
    for label, history in histories.items():
        for ax, key in zip(axes, keys):
            ax.plot(smooth([h[key] for h in history]), label=label)
            ax.set_title(key)
            ax.set_xlabel("step")
    if keys[-1] == "z_std":
        axes[-1].axhline(1 / math.sqrt(D), ls="--", color="gray", label=r"$1/\sqrt{d}$")
    axes[-1].legend()
    plt.tight_layout()
    plt.show()


plot_history({"MoCo K=4096": moco_base.history})

**Q2.1.** What is chance level for `acc_top1`? How far above chance does MoCo get?

*Your answer:*

**Q2.2.** Follow `pos_sim` and `neg_sim` over training. What happens to each of them, and in which order?

*Your answer:*

**Q2.3.** Compare `z_std` with $1/\sqrt d\approx0.088$. Are the queries spread over the sphere?

*Your answer:*

**Q2.4.** How much better is MoCo than the untrained encoder, by linear probe and by kNN? Which of the two gains more, and why?

*Your answer:*

## Exercise 3: Why must the momentum be large?
Unrolling the update $\theta_k^{(t)}=m\,\theta_k^{(t-1)}+(1-m)\,\theta_q^{(t)}$ gives
$$
\theta_k^{(t)}=(1-m)\sum_{s=0}^{t-1}m^{s}\,\theta_q^{(t-s)}+m^{t}\,\theta_k^{(0)},
$$
a weighted average of the query network over roughly the last $1/(1-m)$ steps (the weights $m^s$ halve every $\ln2/(1-m)$ steps).

A key stays in the queue for $K/B$ steps, so the queue holds keys from $K/B$ different versions of the key network. MoCo needs these versions to be almost the same network: a **consistent dictionary**.

**Q3.1.** Complete the cell below. For which values of $m$ is $1/(1-m)$ much larger than $K/B$?

*Your answer:*

In [ ]:
for m in [0.0, 0.9, 0.99, 0.999]:
    horizon = ...  # TODO: the number of steps the EMA averages over
    print(f"m = {m:<6} the EMA averages over ~{horizon:5.0f} steps; a key stays in the queue for {config.MOCO_K // B} steps")

**The experiment.** `KeyEncoderSnapshots` saves the key network after every step and keeps the last $K/B+1$ copies: exactly the networks that computed the keys now in the queue. After training, we encode one fixed batch with every copy and measure
$$\cos\big(k^{(t-a)},\,k^{(t)}\big),$$
how much a key computed $a$ steps ago differs from the key the newest network gives for the same image. It also records $\|\theta_q-\theta_k\|_2$ after every step.

In [ ]:
def key_network(model):
    """Key encoder + key projector as one module (the weights are shared, not copied)."""
    return nn.Sequential(model.key_encoder, model.key_projector)


def flat_params(modules):
    return torch.cat([p.detach().flatten() for module in modules for p in module.parameters()])


class KeyEncoderSnapshots(pl.Callback):
    """Copies of the key network from the last n steps, and ||theta_q - theta_k|| after every step."""

    def __init__(self, n):
        self.n = n
        self.snapshots = []  # oldest first
        self.gap = []

    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
        state = ...  # TODO: a copy of key_network(pl_module).state_dict(), with every tensor cloned
        self.snapshots = (self.snapshots + [state])[-self.n:]
        theta_q = flat_params([pl_module.encoder, pl_module.projector])
        theta_k = flat_params([pl_module.key_encoder, pl_module.key_projector])
        self.gap.append(...)  # TODO: the L2 distance between theta_q and theta_k


@torch.no_grad()
def consistency_curve(model, snapshots, images):
    """Mean cos(key from the network a steps ago, key from the newest network), for a = 0, 1, ..."""
    net = copy.deepcopy(key_network(model)).cpu().eval()
    keys = []
    for state in snapshots:
        net.load_state_dict(state)
        keys.append(F.normalize(net(images), dim=1))
    newest = keys[-1]
    return ...  # TODO: one mean cosine per snapshot, newest snapshot first (age 0)

In [ ]:
(x_fixed, _), _ = next(iter(contrastive_loader(batch_size=256)))  # one fixed batch of views
momentum_runs = {}
for m in [0.0, 0.9, 0.99, 0.999]:
    snapshots = KeyEncoderSnapshots(n=config.MOCO_K // B + 1)
    model = fit(make_moco(m=m), contrastive_loader(), name=f"ex3_m{m}", epochs=1, callbacks=[snapshots])
    momentum_runs[m] = {
        "consistency": consistency_curve(model, snapshots.snapshots, x_fixed),
        "gap": snapshots.gap,
        "history": model.history,
    }
    evaluate(model.encoder, f"MoCo m={m} (1 epoch)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for m, run in momentum_runs.items():
    axes[0].plot(run["consistency"], label=f"m={m}")
    axes[1].plot(run["gap"], label=f"m={m}")
    axes[2].plot(smooth([h["loss"] for h in run["history"]]), label=f"m={m}")
axes[0].set(title="keys in the queue at the end of training", xlabel="age of the key, a (steps)",
            ylabel=r"$\cos(k^{(t-a)}, k^{(t)})$")
axes[1].set(title=r"$\|\theta_q-\theta_k\|_2$", xlabel="step")
axes[2].set(title="training loss", xlabel="step")
for ax in axes:
    ax.legend()
plt.tight_layout()
plt.show()

**Q3.2.** Describe the consistency curves. For which $m$ does a key at the back of the queue (age 64) still look like a key from the current network?

*Your answer:*

**Q3.3.** For $m=0$ the key network is a copy of the query network, yet the gap $\|\theta_q-\theta_k\|$ is not exactly zero. Why? How does the gap behave for the other values of $m$, and why?

*Your answer:*

**Q3.4.** The MoCo paper reports that training *fails* with $m=0$ (on ImageNet). What happened here? Look at the loss curves and the accuracies.

*Your answer:*

**Q3.5.** And $m=1$?

*Your answer:*

## Exercise 4: More negatives without a bigger batch
MoCo's main claim: the number of negatives $K$ no longer depends on the batch size. Keep $B=64$ and vary $K$. As a reference, train SimCLR (Lecture 4) with $N=64$ and $N=256$; SimCLR has $2N-2$ negatives per anchor. Every run sees the same images (`EPOCHS` epochs). The $K=4096$ run is `moco_base` from Exercise 2.

**Q4.1. Prediction.** How will accuracy change from $K=256$ to $K=16384$? Will SimCLR with $N=64$ (126 negatives) do as well as MoCo with 4096?

*Your answer:*

In [ ]:
negatives = {"MoCo K=4096": 4096}  # run name -> negatives per query; moco_base is evaluated already
for K in [256, 1024, 16384]:
    model = ...  # TODO: train MoCo with a queue of size K (make_moco and fit)
    # TODO: evaluate(...) under the name f"MoCo K={K}"
    negatives[f"MoCo K={K}"] = K

simclr_models = {}
for N in [64, 256]:
    pl.seed_everything(42, verbose=False)
    model = SimCLR(learning_rate=config.LEARNING_RATE, latent_dim=config.LATENT_DIM, projector_dim=D,
                   temperature=config.TEMPERATURE, margin=config.TRIPLET_MARGIN, loss_name="nt_xent")
    simclr_models[N] = fit(model, contrastive_loader(batch_size=N), name=f"ex4_simclr_N{N}")
    evaluate(simclr_models[N].encoder, f"SimCLR N={N}")
    negatives[f"SimCLR N={N}"] = 2 * N - 2

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, metric in zip(axes, ["linear_probe", "knn"]):
    for method, style in [("MoCo", "o-"), ("SimCLR", "s--")]:
        names = sorted([name for name in negatives if name.startswith(method)], key=negatives.get)
        ax.plot([negatives[name] for name in names], [results[name][metric] for name in names], style, label=method)
    ax.set(xscale="log", xlabel="negatives per query", ylabel="test accuracy", title=metric)
    ax.legend()
plt.tight_layout()
plt.show()

**Q4.2.** Describe the trend for MoCo. Does accuracy keep improving with $K$?

*Your answer:*

**Q4.3.** Compare SimCLR with MoCo. Is "the same number of epochs" a fair comparison? (How many optimizer steps does each run make?)

*Your answer:*

**Q4.4.** These runs used a small encoder on 28×28 images. Which of them would be impossible on a single GPU with ImageNet-size images and a ResNet-50, and why?

*Your answer:*

## Exercise 5: Who are the negatives? (toward SupCon)
MoCo never sees the labels, but Fashion-MNIST has $C=10$ classes. A queue of $K$ random training images contains about $K/C$ images from the query's own class: **false negatives**, which the loss pushes away.

The gradient shows how strongly each negative is pushed. With $p_i=\operatorname{softmax}(\text{logits})_i$,
$$
\frac{\partial\mathcal{L}_q}{\partial q}=\frac{1}{\tau}\Big(\sum_{i=1}^{K}p_i\,k_i-(1-p_+)\,k_+\Big),
$$
so negative $k_i$ is pushed with weight $p_i$. Because of the small $\tau$, the most similar negatives get almost all of the push.

We rebuild a queue with `moco_base`: $K=4096$ training images through the key network (one view each) and 512 other images through the query network, and we keep the labels, for analysis only.

**Q5.1. Prediction.** About how many false negatives are in a queue of 4096? What share of the push $\sum_{i\,\text{same class}}p_i\,/\sum_i p_i$ do you expect them to get?

*Your answer:*

In [ ]:
@torch.no_grad()
def build_queue(model, n_keys=config.MOCO_K, n_queries=512):
    """Queries and a queue of keys from fresh views, with their labels and images."""
    model = model.to(DEVICE).eval()
    batches = iter(contrastive_loader(batch_size=n_queries))
    (x_q, _), y_q = next(batches)
    q = F.normalize(model.projector(model.encoder(x_q.to(DEVICE))), dim=1).cpu()
    keys, labels, images = [], [], []
    while sum(len(y) for y in labels) < n_keys:
        (_, x_k), y_k = next(batches)
        keys.append(F.normalize(model.key_projector(model.key_encoder(x_k.to(DEVICE))), dim=1).cpu())
        labels.append(y_k)
        images.append(x_k)
    return q, y_q, x_q, torch.cat(keys)[:n_keys], torch.cat(labels)[:n_keys], torch.cat(images)[:n_keys]


q, y_q, x_q, keys, y_keys, x_keys = build_queue(moco_base)
sim = q @ keys.T                                        # [512, K] cosine similarities
same = y_q[:, None] == y_keys[None, :]                  # [512, K] True = same class (false negative)
p = torch.softmax(sim / config.TEMPERATURE, dim=1)      # push weights; leaving out the positive does not change their ratios
print(f"same-class keys in the queue: {same.float().mean():.3f}   (1/C = {1 / 10:.3f})")

In [ ]:
push_share = ...  # TODO: the share of the push (sum of p) on same-class keys, averaged over queries
hardest = sim.topk(10, dim=1).indices  # the 10 most similar keys of each query
hard_same = ...  # TODO: the fraction of same-class keys among them
print(f"share of the push on same-class keys      : {push_share:.3f}")
print(f"same-class keys among the 10 hardest      : {hard_same:.3f}")

plt.figure(figsize=(7, 3.5))
plt.hist(sim[~same].numpy(), bins=100, density=True, alpha=0.6, label="different class")
plt.hist(sim[same].numpy(), bins=100, density=True, alpha=0.6, label="same class (false negatives)")
plt.xlabel(r"$q\cdot k_i$")
plt.ylabel("density")
plt.legend()
plt.title("Similarity of the queries to the keys in the queue")
plt.show()

rows = []
for i in range(6):
    rows += [x_q[i]] + [x_keys[j] for j in hardest[i, :8]]
show_grid(torch.stack(rows), "query (left) and its 8 hardest negatives", nrow=9)

**Q5.2.** The same-class keys are about 10% of the queue. What share of the push do they get? Explain with the gradient formula.

*Your answer:*

**Q5.3.** Look at the grid of hardest negatives. Are they mistakes of the encoder?

*Your answer:*

**Q5.4.** Supervised contrastive learning (SupCon) uses the labels. With $P(i)$ the samples that share the label of anchor $i$ and $A(i)$ all samples except $i$:
$$
\mathcal{L}^{\text{sup}}_i=-\frac{1}{|P(i)|}\sum_{p\in P(i)}\log\frac{\exp(z_i\cdot z_p/\tau)}{\sum_{a\in A(i)}\exp(z_i\cdot z_a/\tau)}
$$
How would SupCon treat the keys in the grid above? What happens to the share you measured?

*Your answer:*

**Q5.5.** Without labels, how could we reduce the damage of false negatives? And why is this a bigger problem on Fashion-MNIST than on ImageNet?

*Your answer:*

## Exercise 6: Audit the augmentations (MNIST)
In Lecture 3 we asked whether an augmentation preserves the label, $y(t(x))=y(x)$. Contrastive learning makes the encoder invariant to whatever $\mathcal{T}$ changes. Read `build_simclr_transform` in `augmentations.py`.

**Q6.1.** Which steps of `build_simclr_transform` **(a)** do nothing on 1-channel images, and **(b)** are questionable for digits?

*Your answer:*

Build two alternatives and compare three policies with the same MoCo settings, on MNIST.

In [ ]:
def mnist_view(flip, crop_scale):
    """build_simclr_transform, with the flip and the crop scale as arguments."""
    steps = [
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        # TODO: RandomResizedCrop to 28x28, with scale=crop_scale
    ]
    # TODO: add RandomHorizontalFlip(p=0.5) only when flip is True
    steps += [
        v2.RandomApply([v2.ColorJitter(0.8, 0.8, 0.8, 0.2)], p=0.8),
        v2.RandomGrayscale(p=0.2),
        v2.RandomApply([v2.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0))], p=0.5),
        v2.Normalize((0.1307,), (0.3081,)),
    ]
    return v2.Compose(steps)


policies = {
    "lecture": build_simclr_transform(),
    "no flip": mnist_view(flip=False, crop_scale=(0.08, 1.0)),
    "no flip, crop 0.3-1": mnist_view(flip=False, crop_scale=(0.3, 1.0)),
}

digits = datasets.MNIST(DATA_DIR, train=True, download=True).data[:8]  # uint8, [8, 28, 28]
for name, view in policies.items():
    pairs = [view(digit[None]) for digit in digits for _ in range(2)]  # two views per digit, side by side
    show_grid(torch.stack(pairs), f"{name}: two views of each digit", nrow=16)

**Q6.2. Prediction.** Rank the three policies by linear-probe accuracy on MNIST before you train.

*Your answer:*

In [ ]:
evaluate(make_moco().encoder, "MNIST: random encoder", dataset="MNIST")
for name, view in policies.items():
    model = fit(make_moco(), contrastive_loader("MNIST", view_transform=view), name=f"ex6_{name.replace(' ', '_')}")
    evaluate(model.encoder, f"MNIST: {name}", dataset="MNIST")

**Q6.3.** Which policy won? Was the difference as large as you expected? Explain with what the views look like.

*Your answer:*

**Q6.4.** For Fashion-MNIST, would you keep the horizontal flip?

*Your answer:*

## Exercise 7: Remove the negatives (toward BYOL)
What if we keep only the positive term?
$$
\mathcal{L}_q^{\text{pos}}=-\frac{q\cdot k_+}{\tau}
$$

**Q7.1.** Find weights that make this loss minimal without learning anything about the images. Why does the full MoCo loss not have this problem?

*Your answer:*

In [ ]:
moco_pos_only = ...  # TODO: train make_moco(use_negatives=False) for 1 epoch
evaluate(moco_pos_only.encoder, "MoCo without negatives")
n = len(moco_pos_only.history)
plot_history({"with negatives": moco_base.history[:n], "without negatives": moco_pos_only.history},
             keys=("pos_sim", "neg_sim", "z_std"))

Complete collapse (one point) is the extreme case. A milder failure is **dimensional collapse**: the embeddings vary, but only along a few directions. The singular values of the centered embedding matrix $Z\in\mathbb{R}^{n\times d}$ show how many directions are used.

In [ ]:
@torch.no_grad()
def embedding_spectrum(model, n=2000):
    """Singular values of the centered projector outputs of n test images."""
    model = model.to(DEVICE).eval()
    x, _ = next(iter(plain_loader(train=False, batch_size=n)))
    z = F.normalize(model.projector(model.encoder(x.to(DEVICE))), dim=1).float().cpu()
    z = ...  # TODO: center z (subtract the mean embedding)
    return ...  # TODO: the singular values of z


plt.figure(figsize=(6.5, 3.8))
for label, model in [("random init", make_moco()), ("MoCo", moco_base), ("MoCo without negatives", moco_pos_only)]:
    plt.semilogy(embedding_spectrum(model), label=label)
plt.xlabel("index")
plt.ylabel("singular value")
plt.legend()
plt.title("Spectrum of the projector output (2000 test images)")
plt.show()

**Q7.2.** What happened to `pos_sim`, `neg_sim`, `z_std` and the accuracies without negatives?

*Your answer:*

**Q7.3.** Read the spectra. How many directions does each model use?

*Your answer:*

**Q7.4.** This model already has a momentum encoder. Was it enough to prevent collapse? What will BYOL (next lecture) need to add?

*Your answer:*

## Exercise 8: What did each encoder learn?
All the encoders in `model.py` share the same 64-dimensional convolutional encoder, so we can compare training objectives directly:

| encoder | objective | labels |
|---|---|---|
| random | none | no |
| `AutoEncoder` | reconstruction (MSE) | no |
| `SimCLR`, $N=256$ | NT-Xent, negatives from the batch | no |
| `MoCo`, $K=4096$ | InfoNCE, negatives from the queue | no |
| `SupervisedEncoder` | cross-entropy | yes |

Train the two missing ones (the autoencoder from Lecture 2 and the supervised encoder) for the same number of epochs.

In [ ]:
pl.seed_everything(42, verbose=False)
autoencoder = fit(AutoEncoder(learning_rate=config.LEARNING_RATE, latent_dim=config.LATENT_DIM),
                  plain_loader(train=True, shuffle=True, batch_size=B), name="ex8_autoencoder")
pl.seed_everything(42, verbose=False)
supervised = fit(SupervisedEncoder(learning_rate=config.LEARNING_RATE, num_classes=10, latent_dim=config.LATENT_DIM),
                 plain_loader(train=True, shuffle=True, batch_size=B), name="ex8_supervised")
evaluate(autoencoder.encoder, "autoencoder")
evaluate(supervised.encoder, "supervised")

zoo = {
    "random encoder": make_moco().encoder,
    "autoencoder": autoencoder.encoder,
    "SimCLR N=256": simclr_models[256].encoder,
    "MoCo K=4096": moco_base.encoder,
    "supervised": supervised.encoder,
}
print(f"\n{'encoder':<16}{'linear probe':>14}{'kNN':>8}")
for name in zoo:
    print(f"{name:<16}{results[name]['linear_probe']:>14.3f}{results[name]['knn']:>8.3f}")

**Nearest neighbors.** For one test image of six classes, the eight most similar images among the first 10,000 training images (cosine similarity of the encoder features).

In [ ]:
@torch.no_grad()
def nearest_neighbors(encoder, queries, gallery, k=8):
    """Indices of the k gallery images most similar to each query (cosine similarity)."""
    encoder = encoder.to(DEVICE).eval()
    fq = F.normalize(encoder(queries.to(DEVICE)), dim=1)
    fg = F.normalize(encoder(gallery.to(DEVICE)), dim=1)
    return ...  # TODO: indices of the k largest similarities in each row


test_x, test_y = next(iter(plain_loader(train=False, batch_size=10_000)))
gallery_x, gallery_y = next(iter(plain_loader(train=True, batch_size=10_000)))
pick = [int((test_y == c).nonzero()[0]) for c in [0, 1, 2, 4, 6, 7]]  # one test image per class
queries = test_x[pick]
for name, encoder in zoo.items():
    neighbors = nearest_neighbors(encoder, queries, gallery_x)
    hit_rate = (gallery_y[neighbors] == test_y[pick][:, None]).float().mean()
    rows = []
    for i in range(len(pick)):
        rows += [queries[i]] + [gallery_x[j] for j in neighbors[i]]
    show_grid(torch.stack(rows), f"{name}: query and 8 nearest training images ({hit_rate:.0%} same class)", nrow=9)

**t-SNE** of the encoder features of 2000 test images, colored by the true class (the encoders never saw these labels, except `supervised`).

In [ ]:
from sklearn.manifold import TSNE

classes = getattr(datasets, DATASET).classes
x_sub, y_sub = test_x[:2000], test_y[:2000]
fig, axes = plt.subplots(1, len(zoo), figsize=(4 * len(zoo), 4.4))
for ax, (name, encoder) in zip(axes, zoo.items()):
    feats, _ = extract_features(encoder, [(x_sub, y_sub)])
    emb = TSNE(n_components=2, init="pca", random_state=0).fit_transform(feats.numpy())
    ax.scatter(emb[:, 0], emb[:, 1], c=y_sub, cmap="tab10", s=3)
    ax.set_title(name)
    ax.axis("off")
handles = [plt.Line2D([], [], marker="o", ls="", color=plt.cm.tab10(c), label=label) for c, label in enumerate(classes)]
fig.legend(handles=handles, loc="lower center", ncol=len(classes))
plt.tight_layout(rect=(0, 0.08, 1, 1))
plt.show()

**Q8.1.** Rank the encoders by linear probe and by kNN. Are the two rankings the same?

*Your answer:*

**Q8.2.** In the nearest-neighbor grids, what do the neighbors found by the random encoder and by the autoencoder have in common with the query? What do MoCo's neighbors have in common with it?

*Your answer:*

**Q8.3.** In the t-SNE plots, which classes stay mixed even for the best self-supervised encoder? Why are these hard for an objective that never sees labels?

*Your answer:*

**Q8.4.** The supervised encoder is an upper bound here. Name one situation where you would still prefer the MoCo encoder.

*Your answer:*

## Bonus

**B1.** Evaluate the *key* encoder of `moco_base` (see R8). Is it better or worse than the query encoder?

*Your answer:*

In [ ]:
# TODO: evaluate moco_base.key_encoder
evaluate(moco_base.encoder, "MoCo K=4096, query encoder")

**B2.** MoCo v2 changed $\tau$ from 0.07 to 0.2. Using the gradient formula of Exercise 5, what does a larger $\tau$ do to the push on the hardest negatives? How should that interact with false negatives? (Try it: `config.TEMPERATURE = 0.2` before `make_moco()`.)

*Your answer:*

**B3.** MoCo v3 (for Vision Transformers) removed the queue and uses the other images in a large batch as negatives, like SimCLR. Why might the queue no longer be worth it?

*Your answer:*

**B4.** Exercise 6 showed that milder crops help a lot on MNIST. Does that carry over to Fashion-MNIST? Train MoCo on Fashion-MNIST with the flip kept (Q6.4) and crops of at least 30% of the image, and compare with `moco_base` (linear probe 0.784, kNN 0.765).

*Your answer:*

In [ ]:
view = ...  # TODO: mnist_view with the flip (fine for clothes) and crops of at least 30%
moco_mild = fit(make_moco(), contrastive_loader(view_transform=view), name="b4_fashion_crop0.3")
evaluate(moco_mild.encoder, "MoCo K=4096, crop 0.3-1")

## References
* Caron, M., Touvron, H., Misra, I., Jégou, H., Mairal, J., Bojanowski, P., & Joulin, A. (2021). Emerging properties in self-supervised vision transformers. *ICCV*.
* Chen, T., Kornblith, S., Norouzi, M., & Hinton, G. (2020). A simple framework for contrastive learning of visual representations. *ICML*.
* Chen, X., Fan, H., Girshick, R., & He, K. (2020). Improved baselines with momentum contrastive learning. *arXiv:2003.04297*.
* Chen, X., Xie, S., & He, K. (2021). An empirical study of training self-supervised vision transformers. *ICCV*.
* Chuang, C.-Y., Robinson, J., Lin, Y.-C., Torralba, A., & Jegelka, S. (2020). Debiased contrastive learning. *NeurIPS*.
* Dwibedi, D., Aytar, Y., Tompson, J., Sermanet, P., & Zisserman, A. (2021). With a little help from my friends: Nearest-neighbor contrastive learning of visual representations. *ICCV*.
* Grill, J.-B., Strub, F., Altché, F., Tallec, C., Richemond, P. H., Buchatskaya, E., Doersch, C., Pires, B. A., Guo, Z. D., Azar, M. G., Piot, B., Kavukcuoglu, K., Munos, R., & Valko, M. (2020). Bootstrap your own latent: A new approach to self-supervised learning. *NeurIPS*.
* He, K., Fan, H., Wu, Y., Xie, S., & Girshick, R. (2020). Momentum contrast for unsupervised visual representation learning. *CVPR*.
* Huynh, T., Kornblith, S., Walter, M. R., Maire, M., & Khademi, M. (2022). Boosting contrastive self-supervised learning with false negative cancellation. *WACV*.
* Jing, L., Vincent, P., LeCun, Y., & Tian, Y. (2022). Understanding dimensional collapse in contrastive self-supervised learning. *ICLR*.
* Khosla, P., Teterwak, P., Wang, C., Sarna, A., Tian, Y., Isola, P., Maschinot, A., Liu, C., & Krishnan, D. (2020). Supervised contrastive learning. *NeurIPS*.
* Li, J., Zhou, P., Xiong, C., & Hoi, S. C. H. (2021). Prototypical contrastive learning of unsupervised representations. *ICLR*.
* van der Maaten, L., & Hinton, G. (2008). Visualizing data using t-SNE. *Journal of Machine Learning Research, 9*, 2579–2605.
* Wang, F., & Liu, H. (2021). Understanding the behaviour of contrastive loss. *CVPR*.